# MedGemma 4B Clinical Evidence Extractor

Reads each patient's clinical note and extracts whether each policy leaf criterion is met,
along with verbatim evidence and source file reference.

Output: `evidence_extraction/{uuid}_evidence.json` — one file per patient, ready for `policy_tree.py`.

In [ ]:
# Cell 1 — Setup & Authentication
!pip install -q transformers torch accelerate huggingface_hub

import os

print("="*60)
print("MedGemma requires accepting terms at:")
print("  https://huggingface.co/google/medgemma-4b-it")
print()
print("Then authenticate via:")
print("  huggingface-cli login")
print("or set the HF_TOKEN environment variable.")
print("="*60)

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    print("\nWARNING: HF_TOKEN not set. Model loading will fail unless you ran 'huggingface-cli login'.")
else:
    print(f"\nHF_TOKEN found (length={len(HF_TOKEN)}).")

In [ ]:
# Cell 2 — Configuration
from pathlib import Path

MODEL_ID   = "google/medgemma-4b-it"
TREE_PATH  = "rheumatoid_arthritis_initial_auth_decision_tree.json"
NOTES_ROOT = Path("clinical_notes")
OUTPUT_DIR = Path("evidence_extraction")
REGIONS    = [
    "montana_clinical_notes",
    "new_mexico_clinical_notes",
    "wyoming_clinical_notes",
]

print(f"Model     : {MODEL_ID}")
print(f"Tree      : {TREE_PATH}")
print(f"Notes root: {NOTES_ROOT}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Regions   : {REGIONS}")

In [ ]:
# Cell 3 — Load Policy Criteria
import sys
sys.path.insert(0, ".")

from policy_tree import load_tree, get_all_criteria

tree = load_tree(TREE_PATH)
leaf_nodes = get_all_criteria(tree)  # {criterion_id: PolicyNode}

# Build a flat criteria list for the prompt
criteria = [
    {
        "id": cid,
        "source_text": node.source_text or node.summary or node.name,
        "negated": node.negated,
    }
    for cid, node in leaf_nodes.items()
]

print(f"Loaded {len(criteria)} leaf criteria from {TREE_PATH}")
print()
for c in criteria:
    neg_tag = "  [NEGATED]" if c["negated"] else ""
    print(f"  {c['id'][:80]}{neg_tag}")

In [ ]:
# Cell 4 — Load & Index Patient Notes
import re

UUID_PATTERN = re.compile(
    r"^(?P<name>.+)_(?P<uuid>[a-f0-9]{8}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{4}-[a-f0-9]{12})\.txt$"
)

patients = {}  # {uuid: {name, region, files, text}}

for region in REGIONS:
    region_dir = NOTES_ROOT / region
    if not region_dir.exists():
        print(f"WARNING: Region directory not found: {region_dir}")
        continue
    for txt_file in sorted(region_dir.glob("*.txt")):
        m = UUID_PATTERN.match(txt_file.name)
        if not m:
            print(f"WARNING: Could not parse filename: {txt_file.name}")
            continue
        uuid        = m.group("uuid")
        patient_name = m.group("name").replace("_", " ")
        rel_path    = str(txt_file.relative_to(NOTES_ROOT))

        if uuid not in patients:
            patients[uuid] = {
                "name"   : patient_name,
                "uuid"   : uuid,
                "region" : region,
                "files"  : [],
                "texts"  : [],
            }
        patients[uuid]["files"].append(rel_path)
        patients[uuid]["texts"].append(txt_file.read_text(encoding="utf-8"))

# Merge multi-file text (rare, but possible)
for p in patients.values():
    p["text"] = "\n\n---\n\n".join(p["texts"])
    del p["texts"]

print(f"Indexed {len(patients)} patients across {len(REGIONS)} regions.")

# Sanity check: show region distribution
from collections import Counter
region_counts = Counter(p["region"] for p in patients.values())
for r, n in sorted(region_counts.items()):
    print(f"  {r}: {n} patients")

In [ ]:
# Cell 5 — Load MedGemma 4B
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Determine best available device
if torch.cuda.is_available():
    device_label = "CUDA"
elif torch.backends.mps.is_available():
    device_label = "MPS (Apple Silicon)"
else:
    device_label = "CPU"
print(f"Device: {device_label}")

load_kwargs = dict(
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
if HF_TOKEN:
    load_kwargs["token"] = HF_TOKEN

print(f"Loading tokenizer from {MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN if HF_TOKEN else None)

print(f"Loading model from {MODEL_ID} (bfloat16, device_map=auto) ...")
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **load_kwargs)
model.eval()

print("Model loaded successfully.")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        mem = torch.cuda.memory_allocated(i) / 1e9
        print(f"  GPU {i} memory allocated: {mem:.2f} GB")

In [ ]:
# Cell 6 — Define Prompt + Inference Function
import json


def _build_criteria_block(criteria: list[dict]) -> str:
    """Format criteria as a numbered list for the prompt."""
    lines = []
    for i, c in enumerate(criteria, 1):
        neg_hint = (
            " [NEGATED — met=true means the note confirms this negative condition is satisfied]"
            if c["negated"] else ""
        )
        lines.append(f"{i}. ID: {c['id']}\n   Criterion: {c['source_text']}{neg_hint}")
    return "\n".join(lines)


def _build_prompt(note_text: str, criteria: list[dict]) -> str:
    criteria_block = _build_criteria_block(criteria)
    return (
        "<start_of_turn>user\n"
        "You are a clinical documentation analyst evaluating a patient's eligibility for "
        "Prior Authorization of Adalimumab.\n\n"
        "---\n"
        "## PATIENT CLINICAL NOTE (source of evidence)\n"
        f"{note_text}\n"
        "---\n\n"
        "## YOUR TASK\n"
        "For each policy criterion below, read the PATIENT CLINICAL NOTE above and determine:\n\n"
        "  met:\n"
        "    - true  → the note contains explicit text that satisfies this criterion\n"
        "    - false → the note contains explicit text that CONTRADICTS this criterion\n"
        "    - null  → the note does not mention this topic at all, or is ambiguous\n\n"
        "  evidence:\n"
        "    - Copy a SHORT, VERBATIM PHRASE (max 20 words) from the PATIENT CLINICAL NOTE "
        "that supports your decision.\n"
        "    - *** NEVER copy text from the criterion itself as evidence. ***\n"
        "    - *** NEVER invent or paraphrase. Quote only words that appear in the note. ***\n"
        "    - If no relevant text exists in the note, write exactly: \"Not mentioned in note\"\n\n"
        "  IMPORTANT rules:\n"
        "    1. met=null when the topic is absent or ambiguous — NOT met=false.\n"
        "    2. met=false ONLY when the note explicitly contradicts the criterion "
        "(e.g., note says 'RA resolved' but criterion needs active RA).\n"
        "    3. For drug criteria (e.g. methotrexate failure), met=true only if the note "
        "explicitly states the drug was tried AND failed/stopped.\n\n"
        "## POLICY CRITERIA\n"
        f"{criteria_block}\n\n"
        "## OUTPUT FORMAT\n"
        "Respond ONLY with a JSON array. No markdown fences. No explanation. "
        "Start with [ and end with ].\n"
        "[\n"
        '  {"criterion_id": "...", "met": true/false/null, "evidence": "..."},\n'
        "  ...\n"
        "]\n"
        "<end_of_turn>\n"
        "<start_of_turn>model\n"
    )


def _run_inference(prompt: str, max_new_tokens: int = 3000) -> str:
    """Run the model and return raw generated text."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def _parse_json_response(raw: str) -> list[dict] | None:
    """Extract and parse the JSON array from model output."""
    raw = raw.strip()
    raw = re.sub(r"^```(?:json)?\s*\n?", "", raw, flags=re.MULTILINE)
    raw = re.sub(r"\n?```\s*$", "", raw, flags=re.MULTILINE)
    raw = raw.strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass

    match = re.search(r"\[.*\]", raw, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            pass

    return None


def extract_criteria(
    patient: dict,
    criteria: list[dict],
    model,
    tokenizer,
) -> dict:
    """Run MedGemma on a single patient note and return the evidence dict."""
    note_text   = patient["text"]
    source_file = patient["files"][0] if patient["files"] else "unknown"

    prompt = _build_prompt(note_text, criteria)
    raw    = _run_inference(prompt)
    parsed = _parse_json_response(raw)

    # Retry once with explicit JSON repair instruction
    if parsed is None:
        repair_prompt = (
            prompt
            + raw
            + "\n<end_of_turn>\n<start_of_turn>user\n"
            + "Your previous response was not valid JSON or was incomplete. "
            + "Output ONLY the complete JSON array starting with [ and ending with ].\n"
            + "<end_of_turn>\n<start_of_turn>model\n"
        )
        raw2   = _run_inference(repair_prompt)
        parsed = _parse_json_response(raw2)

    if parsed is not None:
        # Force-inject the real source path and backfill any missing criteria
        found_ids = {item.get("criterion_id") for item in parsed if isinstance(item, dict)}
        for item in parsed:
            if isinstance(item, dict):
                item["source_ref"] = source_file
        for c in criteria:
            if c["id"] not in found_ids:
                parsed.append({
                    "criterion_id": c["id"],
                    "met": None,
                    "evidence": "Missing from model response",
                    "source_ref": source_file,
                })
        criteria_list = parsed
    else:
        criteria_list = [
            {
                "criterion_id": c["id"],
                "met": None,
                "evidence": f"Parse failed. Raw: {raw[:150]}",
                "source_ref": source_file,
            }
            for c in criteria
        ]

    return {
        "patient_id"   : patient["name"].replace(" ", "_"),
        "uuid"         : patient["uuid"],
        "region"       : patient["region"],
        "source_files" : patient["files"],
        "criteria"     : criteria_list,
    }


# --- Quick smoke test on a single patient ---
TEST_UUID = "31d9fa9f-0639-05ea-eaff-5bf8aa742139"  # Ardella_Rosella_Cartwright
if TEST_UUID in patients:
    pat = patients[TEST_UUID]
    print(f"Running smoke test on patient: {pat['name']}")
    test_result = extract_criteria(pat, criteria, model, tokenizer)
    print(f"Extracted {len(test_result['criteria'])} criteria entries.")
    print(json.dumps(test_result['criteria'], indent=2))
else:
    print(f"Test UUID {TEST_UUID} not found. Skipping smoke test.")

In [ ]:
# Cell 7 — Run Extraction Loop (with checkpointing)
from tqdm.auto import tqdm

OUTPUT_DIR.mkdir(exist_ok=True)

skipped   = 0
processed = 0
failed    = 0

for uuid, patient in tqdm(patients.items(), desc="Extracting evidence", unit="patient"):
    out_path = OUTPUT_DIR / f"{uuid}_evidence.json"

    # Idempotent: skip already-processed patients
    if out_path.exists():
        skipped += 1
        continue

    try:
        result = extract_criteria(patient, criteria, model, tokenizer)
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2)
        processed += 1
    except Exception as exc:
        failed += 1
        err_path = OUTPUT_DIR / f"{uuid}_evidence_error.json"
        with open(err_path, "w", encoding="utf-8") as f:
            json.dump({
                "patient_id" : patient["name"],
                "uuid"       : uuid,
                "error"      : str(exc),
            }, f, indent=2)
        tqdm.write(f"ERROR [{uuid}]: {exc}")

print()
print(f"Done. processed={processed}  skipped={skipped}  failed={failed}")

In [ ]:
# Cell 8 — Aggregate + Summary
from policy_tree import load_tree, get_all_criteria, CriterionResult, get_status
from collections import Counter
import glob as _glob

# ------------------------------------------------------------------
# Combine all per-patient JSON files → all_patients.jsonl
# ------------------------------------------------------------------
evidence_files = sorted(OUTPUT_DIR.glob("*_evidence.json"))
# Exclude error files
evidence_files = [p for p in evidence_files if not p.stem.endswith("_evidence_error")]

jsonl_path = OUTPUT_DIR / "all_patients.jsonl"
with open(jsonl_path, "w", encoding="utf-8") as out_f:
    for fp in evidence_files:
        with open(fp, encoding="utf-8") as in_f:
            record = json.load(in_f)
        out_f.write(json.dumps(record) + "\n")

print(f"Wrote {len(evidence_files)} records to {jsonl_path}")

# ------------------------------------------------------------------
# Stats: met / not_met / null counts across all patients
# ------------------------------------------------------------------
met_counter = Counter()  # {"met": N, "not_met": N, "null": N}
total_patients = 0

with open(jsonl_path, encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        total_patients += 1
        for c in record.get("criteria", []):
            val = c.get("met")
            if val is True:
                met_counter["met"] += 1
            elif val is False:
                met_counter["not_met"] += 1
            else:
                met_counter["null"] += 1

print()
print(f"Total patients : {total_patients}")
print(f"Criteria met   : {met_counter['met']}")
print(f"Criteria not_met: {met_counter['not_met']}")
print(f"Criteria null  : {met_counter['null']}")

# ------------------------------------------------------------------
# End-to-end demo: load one patient result → policy_tree.get_status()
# ------------------------------------------------------------------
print()
print("--- End-to-end demo ---")

tree = load_tree(TREE_PATH)

# Pick the first successfully processed patient
demo_file = evidence_files[0] if evidence_files else None
if demo_file:
    with open(demo_file, encoding="utf-8") as f:
        demo = json.load(f)

    # Reconstruct results dict for policy_tree
    results = {}
    for c in demo["criteria"]:
        cid = c["criterion_id"]
        results[cid] = CriterionResult(
            criterion_id=cid,
            met=c.get("met"),
            evidence=c.get("evidence", ""),
            source_ref=c.get("source_ref", ""),
        )

    status = get_status(tree, results)
    print(f"Patient     : {demo['patient_id']}")
    print(f"Overall     : {status.overall}")
    print(f"Met         : {status.met_count}/{status.total_count}")
    print(f"Pending     : {status.pending_count}")
    print()
    print("Criteria breakdown:")
    for c in status.criteria:
        print(f"  [{c['status']:8s}] {c['id'][:70]}")
else:
    print("No evidence files found. Run Cell 7 first.")